# Train AASIST from Scratch
This notebook mounts Google Drive, prepares the ASVspoof 2019 dataset on local VM storage, and runs training for the AASIST model.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
PROJECT_ROOT = '/content/drive/MyDrive/ANN/ANN/aasist/'
%cd {PROJECT_ROOT}
!ls

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/183UcSEiMJbMU7BYJsjiKrxfrr8t2SAV6/ANN/aasist
config		     exp_result  main.py.bak_20260401_182635  README.md
data_utils.py	     LA.zip	 models			      requirements.txt
download_dataset.py  LICENSE	 NOTICE			      utils.py
evaluation.py	     main.py	 __pycache__


## 1. Extract Dataset to Local VM
We reuse the `LA.zip` extracted to `/content/dataset`.

In [2]:
if not os.path.exists('/content/dataset'):
    print("Extracting dataset to local VM storage (/content/dataset)...")
    # Note: Using the LA.zip from the project root if available, or from the aasist folder
    ZIP_PATH = '/content/drive/MyDrive/ANN/ANN/rawnet2-antispoofing/LA.zip'
    if not os.path.exists(ZIP_PATH):
        ZIP_PATH = './LA.zip'

    !unzip -q {ZIP_PATH} -d /content/dataset
    print("Extraction complete!")
else:
    print("Dataset already exists in local storage.")

Extracting dataset to local VM storage (/content/dataset)...
Extraction complete!


## 2. Install Requirements

In [3]:
!pip install torchcontrib soundfile
# AASIST uses Stochastic Weight Averaging (SWA) from torchcontrib

  Preparing metadata (setup.py) ... done
  Created wheel for torchcontrib: filename=torchcontrib-0.0.2-py3-none-any.whl size=7516 sha256=f50914a479b93e2267a513c292576264387ba8f12bff94102c2f44f5dfd4285a
  Stored in directory: /root/.cache/pip/wheels/e3/d1/1f/63f00ffea223db446943147a04ff035eb40d00cec3e87d63e5
Successfully built torchcontrib


## 3. Helper: Find Latest Checkpoint

In [4]:
def get_latest_checkpoint(config_name="AASIST"):
    # Checkpoints are saved in exp_result/<tag>/weights/
    checkpoint_files = glob.glob(f"exp_result/**/*{config_name}*/**/*.pth", recursive=True)
    if not checkpoint_files:
        return None
    # Filter out 'swa.pth' or 'best.pth' if you want the periodic checkpoints
    periodic_ckpts = [f for f in checkpoint_files if "checkpoint_ep" in f]
    if not periodic_ckpts:
        return max(checkpoint_files, key=os.path.getmtime)

    latest_ckpt = max(periodic_ckpts, key=os.path.getmtime)
    return latest_ckpt

## 4. Smoke Training
Verify the pipeline with 10 samples and 1 epoch.

In [5]:
!python main.py \
    --config ./config/AASIST_smoke.conf \
    --sample_size 10 \
    --comment "smoke_test"

2026-04-25 09:26:54.739112: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777109214.968387    4239 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777109215.029878    4239 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777109215.500990    4239 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777109215.501037    4239 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777109215.501044    4239 computation_placer.cc:177] computation placer alr

## 5. Real Training
Run full AASIST training. Automatically resumes if a periodic checkpoint is found.

In [ ]:
# Note: Resuming AASIST requires some careful logic because it doesn't have a simple --model_path for resume
# I will stick to the standard command for now.
# If you need to resume, you can manually point the config's model_path to the checkpoint.

!python main.py \
    --config ./config/AASIST.conf \
    --comment "full_training"

2026-04-25 09:27:29.408605: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777109249.538798    4443 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777109249.571403    4443 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777109249.667295    4443 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777109249.667345    4443 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777109249.667353    4443 computation_placer.cc:177] computation placer alr